### save_model() (Helper)

Demonstrates the `save_model` function. This is a client-side (cleartext) helper function. It is **not** an FHE circuit and cannot be compiled with `fhe.Compiler`.

`save_model` serializes a model's public parameters (weights, trees, centroids, etc.) to a portable JSON file. Compiled circuits and keys are **not** serialized — recompile after loading.

In [ ]:
from concrete_fhe_toolkit.ml.serialization import save_model
from concrete_fhe_toolkit.ml.classes import FHELinearRegression
import os

# Create a model with known parameters
model = FHELinearRegression(weights=[3, 2], bias=-7)

# Save the model to a JSON file
save_model(model, 'test_save_model.json')

# Verify the file was created and contains valid JSON
assert os.path.exists('test_save_model.json'), 'File was not created'

import json
with open('test_save_model.json') as f:
    data = json.load(f)
assert data['model'] == 'FHELinearRegression'
assert data['params']['weights'] == [3, 2]
assert data['params']['bias'] == -7
print(f'Model saved successfully. Contents: {json.dumps(data, indent=2)}')

# Cleanup
os.remove('test_save_model.json')
print('save_model tests passed!')

### load_model() (Helper)

Demonstrates the `load_model` function. This is a client-side (cleartext) helper function. It is **not** an FHE circuit and cannot be compiled with `fhe.Compiler`.

`load_model` reads a JSON file produced by `save_model` and reconstructs a fresh, uncompiled model instance. Call `compile(inputset)` on the loaded model before predicting.

In [ ]:
from concrete_fhe_toolkit.ml.serialization import save_model, load_model
from concrete_fhe_toolkit.ml.classes import FHELinearRegression, FHEDecisionTree
import os

# Test 1: Save and load a linear regression model
original = FHELinearRegression(weights=[3, 2], bias=-7)
save_model(original, 'test_load_model.json')
loaded = load_model('test_load_model.json')

assert isinstance(loaded, FHELinearRegression), f'Expected FHELinearRegression, got {type(loaded).__name__}'
assert loaded.weights == [3, 2], f'Weights mismatch: {loaded.weights}'
assert loaded.bias == -7, f'Bias mismatch: {loaded.bias}'
print(f'Linear model loaded successfully: weights={loaded.weights}, bias={loaded.bias}')
os.remove('test_load_model.json')

# Test 2: Save and load a decision tree model
tree_dict = {'feature': 0, 'threshold': 5, 'left': 1, 'right': 0}
original_tree = FHEDecisionTree(tree=tree_dict)
save_model(original_tree, 'test_load_tree.json')
loaded_tree = load_model('test_load_tree.json')

assert isinstance(loaded_tree, FHEDecisionTree), f'Expected FHEDecisionTree, got {type(loaded_tree).__name__}'
assert loaded_tree.tree == tree_dict, f'Tree mismatch'
print(f'Tree model loaded successfully: tree={loaded_tree.tree}')
os.remove('test_load_tree.json')

print('load_model tests passed!')